# LLM tokenizer

A tokenizer can be seen as a mapping of words to numbers. Well, not exactly words, but sometimes words, sometimes fragments of words.

In [1]:
%pip install regex


[notice] A new release of pip is available: 25.1.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


Load the tokenizer data from the JSON file.

Two major elements are necessary: a split regex and the vocabulary.

The split regex will be used to split the text into tokens

the vocabulary maps the token strings to their corresponding ids

In [2]:
import json

tokenizer_json_path = "models/Llama-3.2-1B/tokenizer.json"

with open(tokenizer_json_path) as f:
    tokenizer_data = json.load(f)
split = next(filter(lambda t: t["type"] == "Split", tokenizer_data["pre_tokenizer"]["pretokenizers"]))
split_regex = split["pattern"]["Regex"]
vocab = {k.replace("Ġ", " ").encode("utf-8"): v for k, v in tokenizer_data["model"]["vocab"].items()}

display(f"Splitting regex: {split_regex}")
display(f"Vocabulary length: {len(vocab)}")

"Splitting regex: (?i:'s|'t|'re|'ve|'m|'ll|'d)|[^\\r\\n\\p{L}\\p{N}]?\\p{L}+|\\p{N}{1,3}| ?[^\\s\\p{L}\\p{N}]+[\\r\\n]*|\\s*[\\r\\n]+|\\s+(?!\\S)|\\s+"

'Vocabulary length: 128000'

Create the inverse vocabulary dictionary allowing to convert token ids to token text

In [3]:
vocab_inv = {v: k for k, v in vocab.items()}

Let's try to split a text with the regex loaded from the configuration file

In [4]:
import regex

tok_regex = regex.compile(split_regex)

In [5]:
text = "Two possibilities exist: either we are alone in the Universe or we are not. Both are equally terrifying."
tokens = [bytes(t, "utf-8") for t in tok_regex.findall(text)]
tokens

[b'Two',
 b' possibilities',
 b' exist',
 b':',
 b' either',
 b' we',
 b' are',
 b' alone',
 b' in',
 b' the',
 b' Universe',
 b' or',
 b' we',
 b' are',
 b' not',
 b'.',
 b' Both',
 b' are',
 b' equally',
 b' terrifying',
 b'.']

The implementation of the binary pair encoding algorithm.

It breaks text into subwords units (tokens) available in the vocabulary.

The text is first split into individual characters and then successive merges produce the actual tokens

In [6]:
def bpe_encode(
    vocab: dict[bytes, int], input: str, visualise: bool = False
) -> list[int]:
    
    # Split the input character by charactersplit into characters then successive merges produce the actual tokens
    parts = [bytes([b]) for b in input]
    
    while True:
        if visualise:
            print(parts)

        # Iterate over all pairs and find the pair we want to merge the most
        min_idx = None
        min_rank = None
        for i, pair in enumerate(zip(parts[:-1], parts[1:])):
            rank = vocab.get(pair[0] + pair[1])
            if rank is not None and (min_rank is None or rank < min_rank):
                min_idx = i
                min_rank = rank

        # If there were no pairs we could merge, we're done!
        if min_rank is None:
            break
        assert min_idx is not None

        # Otherwise, merge that pair and leave the rest unchanged. Then repeat.
        parts = parts[:min_idx] + [parts[min_idx] + parts[min_idx + 1]] + parts[min_idx + 2 :]

    if visualise:
        print()

    tokens = [vocab[part] for part in parts]
    return tokens

The word `incredible` is transformed in two tokens for `in` and `credible`

In [7]:
bpe_encode(vocab, b"incredible", visualise=True)

[b'i', b'n', b'c', b'r', b'e', b'd', b'i', b'b', b'l', b'e']
[b'in', b'c', b'r', b'e', b'd', b'i', b'b', b'l', b'e']
[b'in', b'c', b're', b'd', b'i', b'b', b'l', b'e']
[b'in', b'c', b're', b'd', b'i', b'b', b'le']
[b'in', b'c', b're', b'd', b'ib', b'le']
[b'in', b'cre', b'd', b'ib', b'le']
[b'in', b'cre', b'd', b'ible']
[b'in', b'cred', b'ible']
[b'in', b'credible']



[258, 59041]

Check inverse lookup

In [8]:
vocab_inv[59041]

b'credible'

Let's try to tokenize the previously used text

In [9]:
encoded = []
for token in tokens:
    enc = bpe_encode(vocab, token, None)
    encoded.extend(enc)

print(encoded)

[11874, 24525, 3073, 25, 3060, 584, 527, 7636, 304, 279, 29849, 477, 584, 527, 539, 13, 11995, 527, 18813, 42351, 13]


And now use this encoded token list to reconstruct the original text. We use the inverse vocabulary which maps a token id to the corresponding text

In [10]:
decoded = b""

for t in encoded:
    decoded += vocab_inv[t]
print(decoded)

b'Two possibilities exist: either we are alone in the Universe or we are not. Both are equally terrifying.'
